In [30]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from transformers import EsmTokenizer, EsmModel

In [31]:
from utils import model

device = torch.device(f'cuda' if torch.cuda.is_available() else 'cpu')

# Load LLAMP model

In [32]:
LLAMP_model = model.LLAMP(hidden_feat = 256, pooling = 'CLS')

Some weights of EsmModel were not initialized from the model checkpoint at Daehun/peptide_tuned_ESM-2 and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [33]:
LLAMP_model.load_state_dict(torch.load('model_weight/LLAMP.pth', map_location=device), strict=False)
LLAMP_model.to(device)
LLAMP_model.eval()

LLAMP(
  (bert): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 480, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 480, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0): EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=480, out_features=480, bias=True)
              (key): Linear(in_features=480, out_features=480, bias=True)
              (value): Linear(in_features=480, out_features=480, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=480, out_features=480, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((480,), eps=1e-05, elementwise_affine=True)
  

# Load Dataset

In [34]:
from utils import dataset

In [35]:
test_df = pd.read_csv('data/DBAASP/test.csv')
test_df

,sequence,label,species
0,AGRQTIAKYLRREIRKRGRKWVIAW,0.000000,Escherichia coli
1,KIAGKIAAIAGKIAKIAGAIAKIAGKIA,0.482921,Escherichia coli
2,FLPGLECVSGKIVPTVFCAITRIC,0.766642,Escherichia coli
3,IRPIIRPIIRPIIRPI,1.204120,Escherichia coli
4,TPFLLVGTQIDLR,0.380211,Escherichia coli
...,...,...,...
3736,GRLRNLIEKAGQNIRGKIQGIGRRIKDILKNLQPRPQV,0.634700,Staphylococcus haemolyticus
3737,FQRYFHRYARFLAKIWKG,1.000000,Staphylococcus haemolyticus
3738,YKRWKKWRSKAKKIL,0.296820,Staphylococcus haemolyticus
3739,IGRHFKRRNSIWGICWF,1.170262,Staphylococcus haemolyticus


In [36]:
genome_feat_dict = torch.load('data/Genomic_featrues/genome_features.pt', map_location=device)

seqs = list(test_df.sequence.astype(str))

genome_feats = dataset.get_features(test_df['species'], genome_feat_dict)

In [37]:
test_dataloader = dataset.data_loader(seqs, genome_feats, torch.as_tensor(test_df['label'], dtype=torch.float32), BATCH_SIZE = 32)

c:\Users\mikuh\anaconda3\envs\LLAMP\lib\site-packages\transformers\tokenization_utils_base.py:2870: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


# Check test performance

In [38]:
from utils import utils

In [39]:
labels = []
preds = []
with torch.no_grad():
    for input_ids, attention_mask, genome_feat, label in (test_dataloader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        genome_feat = genome_feat.to(device)
        label = label.to(device)
        
        output = LLAMP_model(input_ids, attention_mask, genome_feat)
        
        labels.append(label)
        preds.append(output)

    labels = torch.cat(labels)
    labels = labels.cpu().numpy()

    preds = torch.cat(preds)
    preds = preds.cpu().numpy()

In [40]:
utils.compute_metrics(preds, labels)

{'R_squre': 0.5363818772283976,
 'mae': 0.37946296,
 'mse': 0.27164358,
 'rmse': 0.5211944,
 'Pearson Q': 0.7345759020961597}

# Inference for seq about 45 species

In [41]:
genome_feat_dict.keys()

dict_keys(['Escherichia coli', 'Staphylococcus aureus', 'Pseudomonas aeruginosa', 'Salmonella enterica', 'Bacillus subtilis', 'Klebsiella pneumoniae', 'Staphylococcus epidermidis', 'Acinetobacter baumannii', 'Enterococcus faecalis', 'Micrococcus luteus', 'Listeria monocytogenes', 'Bacillus cereus', 'Enterococcus faecium', 'Enterobacter cloacae', 'Streptococcus mutans', 'Streptococcus pyogenes', 'Bacillus megaterium', 'Pseudomonas syringae', 'Streptococcus pneumoniae', 'Proteus mirabilis', 'Klebsiella aerogenes', 'Stenotrophomonas maltophilia', 'Serratia marcescens', 'Shigella dysenteriae', 'Proteus vulgaris', 'Streptococcus agalactiae', 'Aeromonas salmonicida', 'Vibrio parahaemolyticus', 'Listeria innocua', 'Aeromonas hydrophila', 'Pasteurella multocida', 'Klebsiella oxytoca', 'Lactococcus lactis', 'Pectobacterium carotovorum', 'Staphylococcus haemolyticus', 'Vibrio alginolyticus', 'Pseudomonas putida', 'Shigella flexneri', 'Staphylococcus xylosus', 'Vibrio anguillarum', 'Corynebacteri

In [42]:
sequence = 'SSSSSSAAAAARRRRRRRGGGGGGGG'
species = 'Escherichia coli'

In [43]:
tokenizer = EsmTokenizer.from_pretrained('Daehun/peptide_tuned_ESM-2')

def get_inputs(seq, species):
    inputs = tokenizer.batch_encode_plus([seq])
    
    input_id = torch.tensor(inputs['input_ids'])
    attention_mask = torch.tensor(inputs['attention_mask'])
    
    genome_feat = torch.as_tensor(genome_feat_dict[species][0], dtype=torch.float32).unsqueeze(0)
    
    return input_id, attention_mask, genome_feat

input_id, attention_mask, genome_feat = get_inputs(sequence, species)

input_id = input_id.to(device)
attention_mask = attention_mask.to(device)
genome_feat = genome_feat.to(device)

c:\Users\mikuh\anaconda3\envs\LLAMP\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [44]:
with torch.no_grad():
    output = LLAMP_model(input_id, attention_mask, genome_feat)
output = output.cpu().numpy()

In [45]:
print(f"MIC (log uM) about {species} : {output}")
print(f"MIC (uM) about {species} : {10**output}")

MIC (log uM) about Escherichia coli : 1.8837993144989014
MIC (uM) about Escherichia coli : 76.52429099350401


# BV peptide screening (shortlist.csv)

Predict MICs for the M3CAD candidate peptides in `shortlist.csv` against:
- **Lactobacillus jensenii** - vaginal commensal we want to **spare** (want HIGH MIC).
- **BV-associated bacteria** (Gardnerella, Prevotella, etc.) we want to **kill** (want LOW MIC).

**Key caveat:** none of these species are among LLAMP's 45 trained species, and the model has no
Bacteroidetes / obligate anaerobes at all. This section runs a **proxy-only first pass** using the
closest in-model species. Every number here is an extrapolation and is computational triage, not ground
truth - the verification cells below bound how far to trust it. Species with no acceptable proxy
(Prevotella, Megasphaera, Sneathia) require the genome route at the end of the notebook.

In [46]:
import os

STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')

# shortlist.csv lives in the project root; only the Sequence column feeds LLAMP,
# the physicochemical columns are used later for the plausibility cross-check.
shortlist = pd.read_csv('shortlist.csv').dropna(subset=['Sequence']).copy()
shortlist['Sequence'] = shortlist['Sequence'].astype(str).str.strip().str.upper()

def _is_valid(seq):
    return len(seq) > 0 and set(seq) <= STANDARD_AA

bad = shortlist.loc[~shortlist['Sequence'].apply(_is_valid), 'Sequence'].tolist()
if bad:
    print('WARNING: dropping sequences with non-standard residues:', bad)
shortlist = shortlist[shortlist['Sequence'].apply(_is_valid)].reset_index(drop=True)

print(f'{len(shortlist)} valid candidate peptides '
      f'(lengths {shortlist.Sequence.str.len().min()}-{shortlist.Sequence.str.len().max()})')


def predict_mic(seq_list, species_list, batch_size=32):
    """Return predicted MIC in log10(uM) for aligned lists of sequences and (in-model) species."""
    seq_list = list(seq_list)
    feats = dataset.get_features(list(species_list), genome_feat_dict)
    loader = dataset.data_loader(seq_list, feats, torch.zeros(len(seq_list)), BATCH_SIZE=batch_size)
    LLAMP_model.eval()
    out = []
    with torch.no_grad():
        for input_ids, attention_mask, genome_feat, _ in loader:
            p = LLAMP_model(input_ids.to(device), attention_mask.to(device), genome_feat.to(device))
            out.append(p.detach().cpu().reshape(-1))
    return torch.cat(out).numpy()

shortlist.head()

22 valid candidate peptides (lengths 17-19)


,Sequence,Gravy Score,Aliphatic Index,Aromaticity,Instability Index,Alpha Helix Fraction,Beta Helix Fraction,Turn Fraction,Charge at pH 7,Isoelectric Point,Charge Density,Average Hydrophobicity,Hydrophobic Moment,Novelty,pct
0,VLGRMLRILQIVYRAAVWP,0.911,1.589,0.105,29.568,0.316,0.105,0.526,2.733,11.711,0.144,0.911,1.328,Moderately novel (50-80%),54.0
1,AKAIYPQLALWMWVLFKFK,0.689,1.132,0.263,30.742,0.526,0.053,0.526,2.792,10.002,0.147,0.689,0.781,Moderately novel (50-80%),58.0
2,PRIWKWQRPVWFKIKRY,-1.171,0.629,0.294,23.547,0.176,0.118,0.471,5.955,11.745,0.350,-1.171,1.247,Moderately novel (50-80%),63.0
3,AKLRLVVVRPIQVKPEATF,0.400,1.332,0.053,31.468,0.368,0.105,0.474,2.797,11.001,0.147,0.400,0.381,Moderately novel (50-80%),64.0
4,FWFAAKRYWVRFKYVEPW,-0.378,0.433,0.444,13.639,0.278,0.056,0.556,2.759,9.993,0.153,-0.378,0.605,Moderately novel (50-80%),65.0


In [47]:
# Map each real target species to its closest in-model proxy species.
# role: 'spare' = commensal to keep alive (want HIGH MIC)
#       'kill'  = BV-associated bacterium to clear (want LOW MIC)
# NOTE: species that share a proxy get identical predictions (MIC depends only on the
#       genome feature vector), so treat same-proxy rows as one crude estimate.
proxy_map = pd.DataFrame([
    ('Lactobacillus jensenii', 'Lactococcus lactis',         'spare', 'order-level (Lactobacillales)'),
    ('Lactobacillus jensenii', 'Enterococcus faecalis',      'spare', 'order-level (Lactobacillales)'),
    ('Gardnerella vaginalis',  'Corynebacterium glutamicum', 'kill',  'phylum-level (Actinobacteria)'),
    ('Gardnerella vaginalis',  'Micrococcus luteus',         'kill',  'phylum-level (Actinobacteria)'),
    ('Fannyhessea vaginae',    'Micrococcus luteus',         'kill',  'phylum-level (Actinobacteria)'),
    ('Mobiluncus spp.',        'Corynebacterium glutamicum', 'kill',  'phylum-level (Actinobacteria)'),
], columns=['real_species', 'proxy_species', 'role', 'proxy_quality'])

# Species with no acceptable in-model proxy - handled by the genome route at the end.
unsupported = ['Prevotella bivia', 'Megasphaera spp.', 'Sneathia spp.']

missing = [p for p in proxy_map['proxy_species'].unique() if p not in genome_feat_dict]
assert not missing, f'Proxy species not in genome_feat_dict: {missing}'

print('Proxy mapping:')
for _, r in proxy_map.iterrows():
    print(f"  [{r.role:5s}] {r.real_species:24s} -> {r.proxy_species:28s} ({r.proxy_quality})")
print('\nNo in-model proxy (need genome route):', ', '.join(unsupported))

Proxy mapping:
  [spare] Lactobacillus jensenii   -> Lactococcus lactis           (order-level (Lactobacillales))
  [spare] Lactobacillus jensenii   -> Enterococcus faecalis        (order-level (Lactobacillales))
  [kill ] Gardnerella vaginalis    -> Corynebacterium glutamicum   (phylum-level (Actinobacteria))
  [kill ] Gardnerella vaginalis    -> Micrococcus luteus           (phylum-level (Actinobacteria))
  [kill ] Fannyhessea vaginae      -> Micrococcus luteus           (phylum-level (Actinobacteria))
  [kill ] Mobiluncus spp.          -> Corynebacterium glutamicum   (phylum-level (Actinobacteria))

No in-model proxy (need genome route): Prevotella bivia, Megasphaera spp., Sneathia spp.


In [48]:
# Predict MIC for every candidate x unique proxy species, then attach the real-species roles.
proxy_species = sorted(proxy_map['proxy_species'].unique())

grid = shortlist[['Sequence']].merge(pd.Series(proxy_species, name='proxy_species'), how='cross')
grid['mic_log_uM'] = predict_mic(grid['Sequence'], grid['proxy_species'])
grid['mic_uM'] = 10 ** grid['mic_log_uM']

# Expand to real species (a proxy can stand in for several BV bacteria).
results = grid.merge(proxy_map, on='proxy_species', how='inner')
results = results[['Sequence', 'real_species', 'proxy_species', 'proxy_quality',
                   'role', 'mic_log_uM', 'mic_uM']]
print(f'{len(grid)} peptide x proxy predictions -> {len(results)} peptide x real-species rows')
results.head()

c:\Users\mikuh\anaconda3\envs\LLAMP\lib\site-packages\transformers\tokenization_utils_base.py:2870: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


88 peptide x proxy predictions -> 132 peptide x real-species rows


,Sequence,real_species,proxy_species,proxy_quality,role,mic_log_uM,mic_uM
0,VLGRMLRILQIVYRAAVWP,Gardnerella vaginalis,Corynebacterium glutamicum,phylum-level (Actinobacteria),kill,0.491605,3.101737
1,VLGRMLRILQIVYRAAVWP,Mobiluncus spp.,Corynebacterium glutamicum,phylum-level (Actinobacteria),kill,0.491605,3.101737
2,VLGRMLRILQIVYRAAVWP,Lactobacillus jensenii,Enterococcus faecalis,order-level (Lactobacillales),spare,1.031383,10.749371
3,VLGRMLRILQIVYRAAVWP,Lactobacillus jensenii,Lactococcus lactis,order-level (Lactobacillales),spare,0.729079,5.358935
4,VLGRMLRILQIVYRAAVWP,Gardnerella vaginalis,Micrococcus luteus,phylum-level (Actinobacteria),kill,0.448055,2.805790


## Selectivity readout

A good BV therapeutic **spares** *Lactobacillus jensenii* (high MIC) while **killing** the BV bacteria
(low MIC). We use a conservative selectivity index:

`selectivity_index = commensal_MIC_worstcase / BV_MIC_worstcase`

- `commensal_MIC_worstcase` = **minimum** predicted MIC across the L. jensenii proxies (lowest = most harmful to the commensal).
- `BV_MIC_worstcase` = **maximum** predicted MIC across the BV proxies (highest = hardest BV organism to kill).

A large index means the peptide spares the commensal and hits BV even under the worst-case reading.
Absolute BV MIC still matters: a highly selective peptide that is weak against BV is not useful.

In [49]:
# Peptide x real-species MIC table (uM)
pivot = results.pivot_table(index='Sequence', columns='real_species',
                            values='mic_uM', aggfunc='mean')

commensal_mic = results[results.role == 'spare'].groupby('Sequence')['mic_uM'].min()
bv_mic = results[results.role == 'kill'].groupby('Sequence')['mic_uM'].max()

summary = pd.DataFrame({
    'commensal_mic_uM': commensal_mic,   # worst-case: lowest MIC on L. jensenii proxies
    'bv_mic_uM': bv_mic,                 # worst-case: highest MIC on BV proxies
})
summary['selectivity_index'] = summary['commensal_mic_uM'] / summary['bv_mic_uM']
summary = summary.merge(shortlist[['Sequence', 'Novelty', 'pct']], on='Sequence', how='left')
summary = summary.sort_values('selectivity_index', ascending=False).reset_index(drop=True)

pd.set_option('display.float_format', lambda v: f'{v:.3f}')
print('Top candidates by selectivity (spare commensal / kill BV):')
summary.head(10)

Top candidates by selectivity (spare commensal / kill BV):


,Sequence,commensal_mic_uM,bv_mic_uM,selectivity_index,Novelty,pct
0,KFMIYFFVKAFKWALLLFV,5.505,2.511,2.192,Moderately novel (50-80%),73.000
1,RYPIWWLKKPFFKKPKALV,5.034,2.323,2.167,Moderately novel (50-80%),67.000
2,LPWVYKVKRAFAPFKIYRQ,14.523,7.252,2.003,Moderately novel (50-80%),74.000
3,FWFAAKRYWVRFKYVEPW,6.583,3.366,1.956,Moderately novel (50-80%),65.000
4,KRIPQVKPRARYFFWWPWK,37.729,21.407,1.762,Moderately novel (50-80%),72.000
5,VLGRMLRILQIVYRAAVWP,5.359,3.102,1.728,Moderately novel (50-80%),54.000
6,KWMWRVQFQRIAFLKWFLK,6.237,4.071,1.532,Moderately novel (50-80%),73.000
7,PQFKLRQWQVKPKRKIPWH,8.661,6.103,1.419,Moderately novel (50-80%),73.000
8,QYPIFAYKWYVQIILRIKY,19.980,14.172,1.410,Moderately novel (50-80%),75.000
9,KAFWWQVYILVVKKRPLFA,8.495,6.031,1.408,Moderately novel (50-80%),67.000


## Verify plausibility

Because the target species are out-of-distribution, we bound trust with four independent checks:

1. **Model baseline** on in-distribution species (test-set R2 / MAE / Pearson from the section above).
2. **Domain-of-applicability**: are the candidate lengths inside the training range?
3. **Literature-AMP controls**: do well-characterized AMPs land within ~1 log10 of published MICs?
4. **Scramble controls**: is each top candidate predicted more potent than its shuffled sequence?
5. **Physicochemical correlation**: does predicted potency track the independent descriptors in `shortlist.csv`?

In [50]:
# (1) Model baseline on in-distribution test set + (2) domain-of-applicability on lengths
train_df = pd.read_csv('data/DBAASP/train.csv')
train_len = train_df['sequence'].astype(str).str.len()
cand_len = shortlist['Sequence'].str.len()

lo, hi = train_len.quantile(0.01), train_len.quantile(0.99)
in_range = ((cand_len >= lo) & (cand_len <= hi)).sum()

print('Training peptide length: min=%d  max=%d  median=%d' %
      (train_len.min(), train_len.max(), train_len.median()))
print('Candidate  peptide length: min=%d  max=%d' % (cand_len.min(), cand_len.max()))
print('Candidates within 1-99%% training length [%.0f, %.0f]: %d / %d'
      % (lo, hi, in_range, len(cand_len)))
print('\nReminder: test-set metrics above (R2 ~0.54, Pearson ~0.73) are for IN-distribution '
      'species; treat BV/commensal numbers as strictly less reliable.')

Training peptide length: min=6  max=50  median=17
Candidate  peptide length: min=17  max=19
Candidates within 1-99% training length [7, 45]: 22 / 22

Reminder: test-set metrics above (R2 ~0.54, Pearson ~0.73) are for IN-distribution species; treat BV/commensal numbers as strictly less reliable.


In [51]:
# (3) Literature-AMP controls: predictions should be near published MIC ranges on IN-model species.
# We score each control by the log10 distance from the NEAREST edge of the published range
# (0 = inside range) and compare it to the model's own test-set RMSE, rather than eyeballing.
lit_controls = pd.DataFrame([
    ('Magainin-2', 'GIGKFLHSAKKFGKAFVGEIMNS',              'Escherichia coli',      32,  64),
    ('Magainin-2', 'GIGKFLHSAKKFGKAFVGEIMNS',              'Staphylococcus aureus', 32, 128),
    ('LL-37',      'LLGDFFRKSKEKIGKEFKRIVQRIKDFLRNLVPRTES', 'Escherichia coli',       5,  32),
    ('Melittin',   'GIGAVLKVLTTGLPALISWIKRKRQQ',           'Escherichia coli',       1,   8),
    ('Pexiganan',  'GIGKFLKKAKKFGKAFVKILKK',               'Escherichia coli',       4,  16),
], columns=['name', 'Sequence', 'species', 'pub_low', 'pub_high'])

lit_controls['pred_MIC_uM'] = 10 ** predict_mic(lit_controls['Sequence'],
                                                lit_controls['species'], batch_size=16)

def _log10_gap(r):
    p = np.log10(r['pred_MIC_uM'])
    lo, hi = np.log10(r['pub_low']), np.log10(r['pub_high'])
    if p < lo:
        return p - lo   # negative -> predicted MORE potent (lower MIC) than literature
    if p > hi:
        return p - hi   # positive -> predicted LESS potent than literature
    return 0.0

TEST_RMSE_LOG = 0.52  # from the in-distribution test set (Check test performance section)
lit_controls['log10_gap'] = lit_controls.apply(_log10_gap, axis=1)
lit_controls['fold_off'] = 10 ** lit_controls['log10_gap'].abs()
lit_controls['consistent'] = lit_controls['log10_gap'].abs() <= 1.0
lit_controls['published_MIC_uM'] = (lit_controls['pub_low'].astype(str)
                                    + '-' + lit_controls['pub_high'].astype(str))

print('Model test-set RMSE = %.2f log10 (a factor of %.1fx). Pass bar: |log10_gap| <= 1.'
      % (TEST_RMSE_LOG, 10 ** TEST_RMSE_LOG))
print('log10_gap sign: negative = model predicts MORE potent than literature (optimistic).')
print('Consistent controls: %d / %d | max gap %.2f log10 | mean signed bias %+.2f log10 (%.2fx)'
      % (lit_controls['consistent'].sum(), len(lit_controls),
         lit_controls['log10_gap'].abs().max(),
         lit_controls['log10_gap'].mean(), 10 ** abs(lit_controls['log10_gap'].mean())))
lit_controls[['name', 'species', 'published_MIC_uM', 'pred_MIC_uM',
              'log10_gap', 'fold_off', 'consistent']]

Literature controls (pass = predicted within ~1 log10 of the published range):


c:\Users\mikuh\anaconda3\envs\LLAMP\lib\site-packages\transformers\tokenization_utils_base.py:2870: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


,name,Sequence,species,published_MIC_uM,pred_MIC_uM
0,Magainin-2,GIGKFLHSAKKFGKAFVGEIMNS,Escherichia coli,~32-64,18.268
1,Magainin-2,GIGKFLHSAKKFGKAFVGEIMNS,Staphylococcus aureus,~32-128,19.637
2,LL-37,LLGDFFRKSKEKIGKEFKRIVQRIKDFLRNLVPRTES,Escherichia coli,~5-32,7.006
3,Melittin,GIGAVLKVLTTGLPALISWIKRKRQQ,Escherichia coli,~1-8,4.003
4,Pexiganan,GIGKFLKKAKKFGKAFVKILKK,Escherichia coli,~4-16,1.194


In [52]:
# (4) Scramble controls: a real AMP signal should be lost when residues are shuffled.
# Compare top candidates vs a shuffled version on a representative BV proxy.
import random
rng = random.Random(42)

def scramble(seq):
    chars = list(seq)
    rng.shuffle(chars)
    return ''.join(chars)

bv_proxy = 'Corynebacterium glutamicum'   # representative BV (Gardnerella) proxy
top_seqs = summary.head(5)['Sequence'].tolist()

scr = pd.DataFrame({'Sequence': top_seqs})
scr['scrambled'] = scr['Sequence'].apply(scramble)
scr['orig_MIC_uM'] = 10 ** predict_mic(scr['Sequence'], [bv_proxy] * len(scr))
scr['scrambled_MIC_uM'] = 10 ** predict_mic(scr['scrambled'], [bv_proxy] * len(scr))
scr['pass (orig more potent)'] = scr['orig_MIC_uM'] < scr['scrambled_MIC_uM']
print(f'Scramble control vs {bv_proxy}:')
scr

c:\Users\mikuh\anaconda3\envs\LLAMP\lib\site-packages\transformers\tokenization_utils_base.py:2870: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
c:\Users\mikuh\anaconda3\envs\LLAMP\lib\site-packages\transformers\tokenization_utils_base.py:2870: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None 

Scramble control vs Corynebacterium glutamicum:


,Sequence,scrambled,orig_MIC_uM,scrambled_MIC_uM,pass (orig more potent)
0,KFMIYFFVKAFKWALLLFV,LAYAFLFFWLFFKMVVKKI,2.511,22.132,True
1,RYPIWWLKKPFFKKPKALV,PKIFFAYKWVPRWPKKKLL,2.323,3.561,True
2,LPWVYKVKRAFAPFKIYRQ,KFQKFWYAVPIKYLPRRAV,7.252,6.233,False
3,FWFAAKRYWVRFKYVEPW,FFWVRWKEYRVAPAKYWF,3.366,13.181,True
4,KRIPQVKPRARYFFWWPWK,AWYFRKFPPKRQKWWPVIR,21.407,1.589,False


In [53]:
# (5) Physicochemical correlation using shortlist.csv's own descriptors (computed independently of LLAMP).
# Lower mic_log = more potent. Cationic / amphipathic peptides should trend toward lower MIC,
# i.e. NEGATIVE correlation for charge-related descriptors is the expected AMP direction.
phys_cols = ['Charge at pH 7', 'Charge Density', 'Hydrophobic Moment',
             'Gravy Score', 'Aromaticity', 'Aliphatic Index']
phys_cols = [c for c in phys_cols if c in shortlist.columns]

bv_log = (results[results.role == 'kill']
          .groupby('Sequence')['mic_log_uM'].mean()
          .rename('bv_mic_log_uM'))
corr_df = shortlist.merge(bv_log, on='Sequence')
corr = (corr_df[phys_cols + ['bv_mic_log_uM']]
        .corr(method='spearman')['bv_mic_log_uM']
        .drop('bv_mic_log_uM')
        .sort_values())

print('Spearman correlation of shortlist descriptors vs predicted BV MIC (log uM):')
print('negative -> higher descriptor value gives lower MIC (more potent) = expected for charge\n')
corr

Spearman correlation of shortlist descriptors vs predicted BV MIC (log uM):
negative -> higher descriptor value gives lower MIC (more potent) = expected for charge



Aromaticity          -0.479
Hydrophobic Moment   -0.269
Charge Density       -0.003
Charge at pH 7        0.032
Gravy Score           0.134
Aliphatic Index       0.383
Name: bv_mic_log_uM, dtype: float64

## Genome route for the real target species (follow-up)

The proxy pass cannot represent Prevotella / Megasphaera / Sneathia at all, and only crudely
approximates the others. To score the **real** species, give LLAMP the actual genome k-mer fingerprint:

1. Download nucleotide genome FASTAs from NCBI (Datasets / Assembly) for each target species.
2. Save them into `data/Genomic_featrues/` with the file names in the cell below.
3. Run the cell - it computes the 340-dim feature via `utils.get_genome_feature_from_file`, injects it into
   `genome_feat_dict`, then you re-run the grid / selectivity / verification cells with the real names added
   to `proxy_map` (or a parallel real-species target list).

This machine had SSL/pip issues earlier, so download the FASTAs manually rather than from the notebook.
Requires Biopython in the LLAMP env (`from Bio import SeqIO`, already a lazy import in `utils/utils.py`).

In [54]:
# Runs only when genome FASTAs are present; otherwise it is a no-op with instructions.
from utils import utils as _utils

real_species_fasta = {
    'Lactobacillus jensenii': 'data/Genomic_featrues/Lactobacillus_jensenii.fna',
    'Gardnerella vaginalis':  'data/Genomic_featrues/Gardnerella_vaginalis.fna',
    'Prevotella bivia':       'data/Genomic_featrues/Prevotella_bivia.fna',
    'Fannyhessea vaginae':    'data/Genomic_featrues/Fannyhessea_vaginae.fna',
    'Megasphaera spp.':       'data/Genomic_featrues/Megasphaera.fna',
    'Sneathia spp.':          'data/Genomic_featrues/Sneathia.fna',
}

added = []
for name, path in real_species_fasta.items():
    if os.path.exists(path):
        feat = _utils.get_genome_feature_from_file(path)
        assert feat.shape == (340,), f'{name}: expected (340,), got {feat.shape}'
        genome_feat_dict[name] = [feat]
        added.append(name)

if added:
    print('Added real species to genome_feat_dict:', added)
    print('Now add these names to a target list and re-run the grid / selectivity / verification cells,')
    print('e.g. predict_mic(shortlist.Sequence, ["Gardnerella vaginalis"] * len(shortlist)).')
else:
    print('No genome FASTAs found - this cell is a no-op until you add them.')
    print('Expected files (download nucleotide genomes from NCBI):')
    for p in real_species_fasta.values():
        print('  ', p)

No genome FASTAs found - this cell is a no-op until you add them.
Expected files (download nucleotide genomes from NCBI):
   data/Genomic_featrues/Lactobacillus_jensenii.fna
   data/Genomic_featrues/Gardnerella_vaginalis.fna
   data/Genomic_featrues/Prevotella_bivia.fna
   data/Genomic_featrues/Fannyhessea_vaginae.fna
   data/Genomic_featrues/Megasphaera.fna
   data/Genomic_featrues/Sneathia.fna


## Export results

In [55]:
os.makedirs('data/M3CAD', exist_ok=True)

results.to_csv('data/M3CAD/mic_predictions.csv', index=False)
summary.to_csv('data/M3CAD/selectivity_ranking.csv', index=False)

print('Wrote:')
print('  data/M3CAD/mic_predictions.csv     (peptide x species predicted MIC, uM)')
print('  data/M3CAD/selectivity_ranking.csv (spare-commensal / kill-BV ranking)')
print()
print('=== Verification summary (proxy-only pass) ===')
print(f'- Candidates screened: {len(shortlist)}')
print(f'- In-model test-set baseline: R2 ~0.54, Pearson ~0.73 (in-distribution species only)')
print(f'- Scramble control pass rate: {int(scr["pass (orig more potent)"].sum())}/{len(scr)} top candidates')
print(f'- Strongest descriptor correlation: {corr.abs().idxmax()} (Spearman {corr[corr.abs().idxmax()]:+.2f})')
print('- CAVEAT: Prevotella/Megasphaera/Sneathia not represented; all target species are')
print('  out-of-distribution. Use the genome route above before reporting real MIC figures.')

Wrote:
  data/M3CAD/mic_predictions.csv     (peptide x species predicted MIC, uM)
  data/M3CAD/selectivity_ranking.csv (spare-commensal / kill-BV ranking)

=== Verification summary (proxy-only pass) ===
- Candidates screened: 22
- In-model test-set baseline: R2 ~0.54, Pearson ~0.73 (in-distribution species only)
- Scramble control pass rate: 3/5 top candidates
- Strongest descriptor correlation: Aromaticity (Spearman -0.48)
- CAVEAT: Prevotella/Megasphaera/Sneathia not represented; all target species are
  out-of-distribution. Use the genome route above before reporting real MIC figures.
